In [ ]:
import pandas as pd
import numpy as np

ROLL_NUMBER = "102303023"

last_two_digits = [int(ROLL_NUMBER[-2]), int(ROLL_NUMBER[-1])]
categories_pool = ["billing", "account", "general"]

d1, d2 = last_two_digits[0], last_two_digits[1]
cat1 = categories_pool[d1 % 3]
cat2 = categories_pool[d2 % 3]

print(f"Roll Number: {ROLL_NUMBER}")
print(f"Last two digits: d1 = {d1} -> d1 % 3 = {d1 % 3} -> Category: '{cat1}'")
print(f"                 d2 = {d2} -> d2 % 3 = {d2 % 3} -> Category: '{cat2}'")

category_faq_templates = {
    "billing": {
        "question": "how can i download my fee payment receipt",
        "answer": "Go to Student Portal > Fee Details > Transaction History and click 'Download Receipt'.",
        "keywords": "receipt invoice transaction download"
    },
    "account": {
        "question": "how do i update my registered mobile number",
        "answer": "Navigate to Profile Settings > Contact Info and verify your new number with an OTP.",
        "keywords": "mobile phone update contact"
    },
    "general": {
        "question": "where can i find the academic calendar",
        "answer": "The academic calendar is published under the 'Academics' tab on the university website.",
        "keywords": "calendar schedule semester dates"
    }
}

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

entry_1 = {
    "question": category_faq_templates[cat1]["question"],
    "answer": category_faq_templates[cat1]["answer"],
    "keywords": category_faq_templates[cat1]["keywords"],
    "category": cat1
}

if cat1 == cat2:
    alt_templates = {
        "billing": ("can i get a refund on tuition fees", "Refund requests must be submitted within 14 days of admission.", "refund fee cancel return"),
        "account": ("how do i enable two-factor authentication", "Go to Security Settings and scan the QR code with Google Authenticator.", "2fa security login verify"),
        "general": ("how do i contact the student helpdesk", "You can email helpdesk@college.edu or call extension 4401.", "helpdesk contact support reach")
    }
    q, a, kw = alt_templates[cat2]
    entry_2 = {"question": q, "answer": a, "keywords": kw, "category": cat2}
else:
    entry_2 = {
        "question": category_faq_templates[cat2]["question"],
        "answer": category_faq_templates[cat2]["answer"],
        "keywords": category_faq_templates[cat2]["keywords"],
        "category": cat2
    }

faq_data = fixed_entries + [entry_1, entry_2]
df_faq = pd.DataFrame(faq_data)

print("\n=== Final 6-Row Personalized Knowledge Base ===")
pd.set_option('display.max_colwidth', None)
display(df_faq)


Roll Number: 102303023
Last two digits: d1 = 2 -> d1 % 3 = 2 -> Category: 'general'
                 d2 = 3 -> d2 % 3 = 0 -> Category: 'billing'

=== Final 6-Row Personalized Knowledge Base ===


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where can i find the academic calendar,The academic calendar is published under the 'Academics' tab on the university website.,calendar schedule semester dates,general
5,how can i download my fee payment receipt,Go to Student Portal > Fee Details > Transaction History and click 'Download Receipt'.,receipt invoice transaction download,billing


In [2]:
import re

def preprocess(text):
    """Lowercase, remove punctuation, and extract unique tokens."""
    tokens = re.findall(r'\b\w+\b', str(text).lower())
    return set(tokens)

def score_query(query: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Scores each FAQ entry against the query string based on keyword and question overlap.
    Returns matching entries ranked by confidence score in descending order.
    """
    query_tokens = preprocess(query)
    if not query_tokens:
        return pd.DataFrame(columns=list(df.columns) + ["score", "confidence_%"])

    scores = []
    for idx, row in df.iterrows():
        kw_tokens = preprocess(row["keywords"])
        q_tokens = preprocess(row["question"])
        
        kw_matches = query_tokens.intersection(kw_tokens)
        q_matches = query_tokens.intersection(q_tokens)
        
        total_overlap = kw_matches.union(q_matches)
        
        if len(total_overlap) == 0:
            raw_score = 0.0
        else:
            base_score = len(total_overlap) / len(query_tokens)
            kw_boost = 1.2 if len(kw_matches) > 0 else 1.0
            raw_score = min(1.0, base_score * kw_boost)
            
        scores.append(round(raw_score, 3))
    
    result_df = df.copy()
    result_df["score"] = scores
    result_df["confidence_%"] = [f"{s * 100:.1f}%" for s in scores]
    
    ranked_df = result_df[result_df["score"] > 0].sort_values(by="score", ascending=False).reset_index(drop=True)
    return ranked_df

test_query = "how can i pay my annual fee online"
print(f"Query: '{test_query}'")
results_q2 = score_query(test_query, df_faq)
display(results_q2[["question", "category", "score", "confidence_%", "answer"]])


Query: 'how can i pay my annual fee online'


,question,category,score,confidence_%,answer
0,how can i pay the fee,billing,0.750,75.0%,"You can pay via UPI, card, or net banking."
1,how can i download my fee payment receipt,billing,0.625,62.5%,Go to Student Portal > Fee Details > Transaction History and click 'Download Receipt'.
2,what is the annual fee,billing,0.300,30.0%,The annual fee is Rs 500.
3,where can i find the academic calendar,general,0.250,25.0%,The academic calendar is published under the 'Academics' tab on the university website.
4,how to reset password,account,0.125,12.5%,Go to Settings > Reset Password.


In [3]:
def same_category(category_name: str, df: pd.DataFrame) -> pd.Series:
    """
    Filters the FAQ DataFrame by category and returns all questions belonging to that category.
    """
    filtered = df[df["category"].str.lower() == category_name.lower().strip()]
    return filtered["question"].reset_index(drop=True)

target_category = cat1
print(f"Querying questions under category: '{target_category}'\n")

questions_in_cat = same_category(target_category, df_faq)
print(f"Total questions found: {len(questions_in_cat)}")
for i, q in enumerate(questions_in_cat, 1):
    print(f"{i}. {q}")


Querying questions under category: 'general'

Total questions found: 2
1. what are your working hours
2. where can i find the academic calendar


In [ ]:
import os

entry_index_to_update = 0
selected_entry = df_faq.iloc[entry_index_to_update]

print(f"Selected Entry (Index {entry_index_to_update}):")
print(f"  Question: '{selected_entry['question']}'")
print(f"  Current Keywords: '{selected_entry['keywords']}'")

default_new_keyword = "tuition"
try:
    user_kw = input(f"Enter a new keyword to append to this entry (Press Enter for '{default_new_keyword}'): ").strip()
    if not user_kw:
        user_kw = default_new_keyword
except (EOFError, NameError):
    user_kw = default_new_keyword

print(f"\nAdding new keyword: '{user_kw}'")

df_updated = df_faq.copy()
current_keywords = df_updated.at[entry_index_to_update, "keywords"]

if user_kw.lower() not in current_keywords.lower().split():
    df_updated.at[entry_index_to_update, "keywords"] = f"{current_keywords} {user_kw}".strip()

print(f"Updated Keywords: '{df_updated.at[entry_index_to_update, 'keywords']}'")

csv_filename = f"{ROLL_NUMBER}_faq_data.csv"
df_updated.to_csv(csv_filename, index=False)

print(f"\nSuccessfully saved updated DataFrame to '{csv_filename}'.")
print("Preview of updated CSV file:")
display(pd.read_csv(csv_filename))


Selected Entry (Index 0):
  Question: 'what is the annual fee'
  Current Keywords: 'fee cost price charge'


In [ ]:

category_counts = df_updated.groupby("category").size().reset_index(name="count")

print("=== FAQ Entries Count Per Category (groupby) ===")
print(category_counts.to_string(index=False))

cat_summary = df_updated.groupby("category").agg(
    total_entries=("question", "count"),
    sample_questions=("question", lambda x: " | ".join(x))
).reset_index()

print("\n=== Detailed Category Summary ===")
display(cat_summary)


In [ ]:
def score_query_tie_aware(query: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Scoring function that checks for ties at the maximum confidence score.
    If multiple entries tie for the top score, it explicitly highlights and prints
    all tied entries for transparency.
    """
    ranked_results = score_query(query, df)
    
    print(f"\n========================================================")
    print(f"Query: \"{query}\"")
    print(f"========================================================")
    
    if ranked_results.empty:
        print("No matching entries found.")
        return ranked_results
    
    max_score = ranked_results["score"].max()
    top_entries = ranked_results[ranked_results["score"] == max_score]
    
    if len(top_entries) > 1:
        print(f"⚠️  TIE DETECTED! {len(top_entries)} entries tied for the top score ({max_score:.3f} / {top_entries.iloc[0]['confidence_%']}):")
        print("-" * 56)
        for i, (_, row) in enumerate(top_entries.iterrows(), 1):
            print(f"  [Tied Top Match #{i}] (Category: {row['category']})")
            print(f"  Question: {row['question']}")
            print(f"  Answer:   {row['answer']}")
            print(f"  Keywords: {row['keywords']}\n")
    else:
        print(f"✅ CLEAR SINGLE WINNER (Confidence: {top_entries.iloc[0]['confidence_%']}):")
        print("-" * 56)
        top = top_entries.iloc[0]
        print(f"  Question: {top['question']}")
        print(f"  Answer:   {top['answer']}")
        print(f"  Category: {top['category']}\n")
        
    print("--- Complete Ranked Results Table ---")
    display(ranked_results[["question", "category", "score", "confidence_%", "answer"]])
    return ranked_results

print("### TEST CASE 1: Query Resulting in a TIE ###")
tie_query = "fee"
res_tie = score_query_tie_aware(tie_query, df_updated)

print("\n\n### TEST CASE 2: Query Resulting in a SINGLE WINNER ###")
unique_query = "how to reset my login password"
res_unique = score_query_tie_aware(unique_query, df_updated)
